### Install required packages

In [1]:
!pip install pandas requests bs4 html5lib lxml plotly -q

In [2]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import warnings
import logging
from datetime import datetime

In [3]:
# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Suppress warnings
warnings.filterwarnings("ignore", category=FutureWarning)

#### 1. Send HTTP Request

In [4]:
# Define the URL for Netflix historical data
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/netflix_data_webpage.html"

In [10]:
try: 
    html_content  = requests.get(url).text
    logger.info("Successfully retrieved data")
except requests.RequestException as e:
    logger.error(f"Failed to retrieve webpage {e}")

2026-06-24 11:56:25,205 - INFO - Successfully retrieved data


#### 2. Parse HTML Content

In [11]:
soup = BeautifulSoup(html_content, 'html.parser')

In [12]:
soup.find('title')

<title>Netflix, Inc. (NFLX) Stock Historical Prices &amp; Data - Yahoo Finance</title>

#### 3. Extract data from HTML Table

In [13]:
netflix_data = pd.DataFrame(columns=["Date", "Open", "High", "Low", "Close", "Volume"])

In [14]:
table_body = soup.find('tbody')

In [ ]:
rows_extracted = 0

for row in table_body.find_all('tr'):
    col = row.find_all("td")

    date = col[0].text.strip()
    Open = col[1].text.strip()
    high = col[2].text.strip()
    low = col[3].text.strip()
    close = col[4].text.strip()
    adj_close = col[5].text.strip()
    volume = col[6].text.strip()
    
    # Append the data of each row to the table
    netflix_data = pd.concat([netflix_data,pd.DataFrame({"Date":[date], "Open":[Open], "High":[high], "Low":[low], "Close":[close], "Adj Close":[adj_close], "Volume":[volume]})], ignore_index=True)    
    rows_extracted += 1

logger.info(f"Successfullt extracted {rows_extracted} rows of Netflix data")

2026-06-24 12:01:56,535 - INFO - Successfullt extracted 70 rows of Netflix data


#### 4. Display Extracted Data

In [16]:
netflix_data.head()

,Date,Open,High,Low,Close,Volume,Adj Close
0,"Jun 01, 2021",504.01,536.13,482.14,528.21,"78,560,600",528.21
1,"May 01, 2021",512.65,518.95,478.54,502.81,"66,927,600",502.81
2,"Apr 01, 2021",529.93,563.56,499.00,513.47,"111,573,300",513.47
3,"Mar 01, 2021",545.57,556.99,492.85,521.66,"90,183,900",521.66
4,"Feb 01, 2021",536.79,566.65,518.28,538.85,"61,902,300",538.85


In [17]:
netflix_data.shape

(70, 7)

In [18]:
netflix_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70 entries, 0 to 69
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Date       70 non-null     object
 1   Open       70 non-null     object
 2   High       70 non-null     object
 3   Low        70 non-null     object
 4   Close      70 non-null     object
 5   Volume     70 non-null     object
 6   Adj Close  70 non-null     object
dtypes: object(7)
memory usage: 4.0+ KB


In [ ]:
# We need to convert columns to numeric and datetime to analyse

netflix_data.describe().T

,count,unique,top,freq
Date,70,70,"Jun 01, 2021",1
Open,70,70,504.01,1
High,70,70,536.13,1
Low,70,70,482.14,1
Close,70,70,528.21,1
Volume,70,70,"78,560,600",1
Adj Close,70,70,528.21,1


#### 5. Data Cleaning and Type Conversion

In [25]:
df_clean = netflix_data.copy()

(a) Date column

In [26]:
df_clean['Date'] = pd.to_datetime(df_clean['Date'], errors='coerce')

In [29]:
df_clean['Year'] = df_clean['Date'].dt.year
df_clean['Month'] = df_clean['Date'].dt.month
df_clean['Day'] = df_clean['Date'].dt.day
df_clean['DayOfWeek'] = df_clean['Date'].dt.dayofweek

(b) Price Columns

In [31]:
price_columns = ['Open', 'High', 'Low', 'Close', 'Adj Close']

In [ ]:
# Convert price columns (remove commas and convert to float)

for col in price_columns:
    df_clean[col] = df_clean[col].str.replace(',', '').astype(float)

In [33]:
# Convert volume (remove commas and convert to int)

if 'Volume' in df_clean.columns:
    df_clean['Volume'] = df_clean['Volume'].str.replace(',', '').astype(int)


In [35]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70 entries, 0 to 69
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       70 non-null     datetime64[ns]
 1   Open       70 non-null     float64       
 2   High       70 non-null     float64       
 3   Low        70 non-null     float64       
 4   Close      70 non-null     float64       
 5   Volume     70 non-null     int64         
 6   Adj Close  70 non-null     float64       
 7   Year       70 non-null     int32         
 8   Month      70 non-null     int32         
 9   Day        70 non-null     int32         
 10  DayOfWeek  70 non-null     int32         
dtypes: datetime64[ns](1), float64(5), int32(4), int64(1)
memory usage: 5.1 KB


In [37]:
df_clean.describe().T

,count,mean,min,25%,50%,75%,max,std
Date,70,2018-07-17 01:42:51.428571392,2015-09-01 00:00:00,2017-02-08 00:00:00,2018-07-16 12:00:00,2019-12-24 06:00:00,2021-06-01 00:00:00,NaN
Open,70.0,280.746,90.41,141.61,292.345,373.875,545.57,145.711699
High,70.0,307.922286,97.48,146.535,330.315,393.3775,593.29,158.539029
Low,70.0,260.706714,79.95,138.36,264.11,342.42,518.28,136.575328
Close,70.0,286.039571,90.03,143.55,294.55,373.2325,540.73,146.445067
Volume,70.0,194787687.142857,61902300.0,117659150.0,169772300.0,242618250.0,497401200.0,98438418.462537
Adj Close,70.0,286.039571,90.03,143.55,294.55,373.2325,540.73,146.445067
Year,70.0,2018.085714,2015.0,2017.0,2018.0,2019.75,2021.0,1.742419
Month,70.0,6.471429,1.0,3.25,6.0,9.75,12.0,3.52104
Day,70.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0


Author: Kriti Tiwari

Date: 24 June, 2026